# Encyclopedic-VQA — Project overview

Answer questions about the entity in an image. Three settings, in increasing order of contribution:

| | Setting | What it is |
|---|---|---|
| **A** | No-RAG | VLM answers from image + question only |
| **B** | Standard RAG | retrieve a Wikipedia article by image, feed its text as context — the **solid baseline to beat** |
| **C** | **Agentic RAG** | our main contribution: an agent that iterates retrieval/reasoning — must beat the best B |

This repo currently builds and tunes **B** (best retrieval strategy). **C** is the next phase.

## Components

| Role | Model / asset | Notes |
|---|---|---|
| VLM (answerer) | Qwen2.5-VL-3B-Instruct | bf16 on GPU |
| Visual encoder | EVA-CLIP-8B | fp16, image + text embeddings |
| Image processor | openai/clip-vit-large-patch14 | 224×224 |
| Knowledge base | encyclopedic_kb_wiki.db (SQLite, ~19 GB) | ~2.0M articles, url → sections; queried on disk, never loaded into RAM |
| Image index | FAISS knn.index (8.9 GB) | ~1.66M vectors, IndexFlatIP (cosine) |
| Test set | encyclopedic_test_subset.json | 1000 examples |

## Notebook layout

- `00_overview` — this file (framing + components + key findings).
- `01_dataset` — test set exploration.
- `02_comparison` — headline comparison (best of each milestone: A / B / C).
- `03_recall_curve` — retrieval recall@k (the bottleneck).
- `strategies/baseline_B` — the full B record (all variants, hit/miss, top-k×top-n, observations).

**Baseline B milestones — BEM accuracy (full 1000-example test set):**

| ID | Strategy | Overall |
|---|---|---|
| A | no-RAG (VLM only) | 0.245 |
| B1 | top-1 + first 3 sections | 0.278 |
| B2.1 | top-5 + all sections (no rerank) | 0.347 |
| B3 | top-5 + CLIP rerank | 0.243 (hurts) |
| B4 | top-5 + cross-encoder | 0.360 |
| B5 | top-20 + cross-encoder + top-n 20 | **0.403** |

Best: **B5** — top-20 + cross-encoder + top-n 20, **BEM 0.403** on the full (unfiltered) KB. Selected by re-running the 10 best ablation configs on the full test set: top-k 10–50 at n≈20 all land ~0.39–0.40 (flat plateau), and top-20/n20 is the robust winner. (Filtering service/short KB paragraphs was tested and dropped — see key findings.) All intermediate variants and analyses in `strategies/baseline_B`.

## Engineering fixes (shared by all RAG strategies)

- **CUDA OOM (16 GB GPU):** EVA-CLIP-8B (~16 GB fp16) + Qwen do not fit on 16 GB → 45 GB GPU (L40S/A40) on `boost_usr_prod` (>24 GB VRAM policy).
- **Host RAM:** the 16 GB KB JSON used to be loaded whole into a dict (~84 GB RSS) → `--mem=128G`. Fixed: the KB is now an on-disk SQLite file (`encyclopedic_kb_wiki.db`) queried by url on demand, so nothing is loaded into RAM → `--mem=32G`. Built once with `src/retrieval/build_kb_sqlite.py`.
- **device-side assert in encode_image:** non-persistent `position_ids` buffers left uninitialized under meta-device / low_cpu_mem_usage loading → re-materialize as `arange` after load.
- **Resume + separate outputs:** predictions keyed by `unique_id`; each strategy writes its own `predictions_*` / `results_*` file.

In [1]:
import json, os

def show(path, label):
    if not os.path.exists(path):
        print('%-24s (pending)' % label); return
    r = json.load(open(path))
    print('%-24s overall %.4f (n=%d)' % (label, r['accuracy_overall'], r['num_examples']))

show('../outputs/baselines/results_A.json', 'A: no-RAG')
show('../outputs/baselines/results_B5.json', 'B best: top20 + cross20')

A: no-RAG                overall 0.2450 (n=1000)
B best: top20 + cross20  overall 0.4030 (n=1000)


## Key findings (baseline B)

- **Retrieval is the primary bottleneck.** When the correct article is retrieved, the VLM answers ~66% of the time (BEM); otherwise accuracy falls to the no-RAG level (~24%). `overall ≈ recall * acc_hit + (1-recall) * 0.24`. recall@5 = 28%, recall@50 = 47%.
- **top-k and top-n must be scaled together, but the plateau is flat.** n=5 is consistently worst (discards the answer paragraph); n≈10–20 is the sweet spot. Across the full test set, top-k 10–50 at n=20 all land ~0.39–0.40 — **top-20/n20 = 0.403** is the robust best, with only marginal spread above the noise floor.
- **Validation (200 ex.) overfits and reshuffles.** Its best config (top-10/n30 = 0.445) drops to 0.386 on the 1000-example test; the robust test winner is top-20/n20. Lesson: pick configs by mechanism and **confirm on the full test set**, never on the small val max.
- **KB context is kept unfiltered (decision).** We tested dropping service sections (References/See also/…) and short stubs (~30% of pooled paragraphs): BEM was unchanged (0.401 filtered ≈ 0.403 full, within noise). We keep the full paragraph set — it confirms the ceiling is *retrieval recall*, not context noise, and it preserves the references/links the agentic system (**C**) may need. Headline B uses the full KB (0.403).
- **Prompt engineering did not help zero-shot.** A short-answer system prompt and a ReAG-style context prompt (a fine-tuning prompt used zero-shot) both collapsed accuracy to ~0.21 (verbose/entity-biased answers). The plain grounded prompt is best.
- **Reranking only matters in that you must not discard good paragraphs.** Cross-encoder (0.360) ≈ give-all-sections (0.347) >> CLIP rerank (0.243). A weak reranker hurts; a good one is no better than dumping everything.
- **By question type:** `automatic` is strong (~0.48); `multi_answer` and `templated` stay weak (~0.29) even at the best config — the hard cases that motivate the agentic pipeline (**C**).
- **Ceiling.** recall@50 ≈ 0.47 caps this approach at ~0.44 BEM; going further needs a better image encoder (higher recall) or an agent that can search the KB by text (**C**).